(lecture5)=
# Transmission lines: Waves in Electrical Circuits

> In this lecture, we will continue to explore the consequence of inductors and capacitors in electrical circuits, but now in a different context: the consequence of the *distributed* inductance and capacitance built into every wire in the lab and in your circuit. How big are these distributed inductances and capacitances? What hapens if we consider these real-life properties of wires? What is the consequence for currents and voltages in our circuit? Here, we will quantitatively address these questions. Starting from a distributed LC model of a wire, we will find that all wires will behave as transmision lines in which currents and voltages propagate as waves at (nearly) the speed of light. We will derive how fast these waves propagate, and by looking at the currenets and voltages that result behave, we will discover the property of the *characteristic impedance* of a transmission line, and how this property affects how waves reflect and transmit as they propagate through the circuit. 


In [2]:
from helper_functions import *
import matplotlib.pyplot as plt
import numpy as np
import schemdraw
import schemdraw.elements as elm

## Inductance and capacitance per unit length

For us so far, the wires in our circuits are "magic tunnels" than instantaneously transmit voltage from one lumped element of our circuit to another. 

As we discussed in the first lecture, all *real* wires have a series resistance, a self-inductance, and a capacitance to ground. The longer the wire, the larger of all these get: a wire is characterised by a resistance per unit length, an inductance per unit length, and a capacitance per unit length. As you saw in the first lecture, the resistance of copper wires can be made *very* small. For that reason, we will start by ignoring the resitance of wires, allowing us to understand how the inductance and capacitance will lead to wave behaviour (we will come back to resistance per unit length at the very end of the lecture). 

If you remember, by using "spherical cow" models, we can get rough estimates of the capacitance per unit length (1 pF/cm) and the inductance (1 nH/mm). As you will see in this lecture, for waves to propagate well, it is important to have a well defined and fixed inductance and capacitance per unit lenght. One way to do this is it work with a [coaxial cable](https://en.wikipedia.org/wiki/Coaxial_cable), which consists of a central conductor, which carries the voltage signal, surrounded by a dielectric and then followed by an outer shield:

In [69]:
show_image("figures/Coaxial_cable_cutaway_PTFE.svg", width_percent=30)

As the signals propagate in the cable, there are currents flowing in the center wire, there are return currents flowing back in the metallic shield, and there is a voltage difference between the core and shield. From electrostatics and magnetostatics, you can calculate the electric and magentic fields that result from these currents and voltages, which will look like this:

In [34]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge, FancyArrowPatch

SKETCH = False          # True -> hand-drawn look (plt.xkcd)
d_over_2, D_over_2 = 1.0, 3.0   # inner conductor radius, shield inner radius
shield_t = 0.35                  # shield thickness
RED, BLUE, GREEN = "#d9362b", "#1f5fe0", "#4caf50"

def arrow(ax, p0, p1, color, rad=0.0, lw=1.6, ms=14):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms,
                                 color=color, lw=lw, connectionstyle=f"arc3,rad={rad}",
                                 shrinkA=0, shrinkB=0))

def build():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5))

    # ---------- Left: physical cross-section ----------
    ax1.add_patch(Wedge((0, 0), D_over_2 + shield_t, 0, 360, width=shield_t,
                        facecolor="white", edgecolor="k", hatch="///", lw=1.5))
    ax1.add_patch(Circle((0, 0), d_over_2, facecolor="white", edgecolor="k", hatch="///", lw=1.5))
    # dimension arrows
    ax1.annotate("", (-d_over_2, -0.0), (d_over_2, 0.0), arrowprops=dict(arrowstyle="<->", lw=1.3))
    ax1.text(0, -0.1, "d", ha="center", va="top", fontsize=22, bbox=dict(fc="white", ec="none", pad=1))
    y = -(D_over_2 + shield_t) - 0.6  # dimension line sits below the shield
    ax1.annotate("", (-D_over_2, y), (D_over_2, y),
                 arrowprops=dict(arrowstyle="<->", lw=1.3))
    ax1.text(0, y - 0.15, "D", ha="center", va="top", fontsize=22)
    for x in (-D_over_2, D_over_2):
        ax1.plot([x, x], [0, y - 0.1], "k", lw=0.8, ls=":")
    # labels
    ax1.annotate("Signal pin\n(center pin)", (0, 0.6), (-2.6, 4.0), color=GREEN, fontsize=20,
                 ha="center", arrowprops=dict(arrowstyle="->", color=GREEN, connectionstyle="arc3,rad=0.3"))
    ax1.annotate("Shield\n(ground)", (2.35, 2.2), (2.7, 4.0), color=GREEN, fontsize=20,
                 ha="center", arrowprops=dict(arrowstyle="->", color=GREEN, connectionstyle="arc3,rad=-0.3"))

    # ---------- Right: fields ----------
    ax2.add_patch(Circle((0, 0), D_over_2, fill=False, ec="k", lw=1.5))
    ax2.add_patch(Circle((0, 0), d_over_2, fill=False, ec="k", lw=1.5))
    # E field: radial (outward from the center pin)
    for a in np.deg2rad(np.arange(10, 360, 360 / 8)):
        u = np.array([np.cos(a), np.sin(a)])
        arrow(ax2, u * d_over_2, u * D_over_2 * 0.97, RED)
    # B field: azimuthal circle with arrows
    rb = (d_over_2 + D_over_2) / 2
    th = np.linspace(0, 2 * np.pi, 400)
    ax2.plot(rb * np.cos(th), rb * np.sin(th), color=BLUE, lw=1.6)
    for a in np.deg2rad(np.arange(0, 360, 45)):
        a0, a1 = a, a + 0.08          # counter-clockwise
        arrow(ax2, rb * np.array([np.cos(a0), np.sin(a0)]),
              rb * np.array([np.cos(a1), np.sin(a1)]), BLUE, ms=16)
    ax2.text(-0.55, D_over_2 * 1.15, r"$\vec{E}$", color=RED, fontsize=30, ha="center", va="center")
    ax2.text(0.45, D_over_2 * 0.8, r"$\vec{B}$", color=BLUE, fontsize=30, ha="center", va="center")

    for ax in (ax1, ax2):
        ax.set_aspect("equal"); ax.axis("off")
    ax1.set_xlim(-4.2, 4.2); ax1.set_ylim(-5, 5)
    ax2.set_xlim(-3.6, 3.6); ax2.set_ylim(-3.6, 4.6)
    fig.tight_layout()
    return fig

fig = build()
show_plot(fig, scale=0.7)

Voltages on center conductor will create radial electric fields, from the voltage between the center conductor, and currents in the center conductor will create magnetic fields that circle around the circumference. Outside the metallic shield, the electric field is screened by charges induced on the outer metal, and the magnetic field is screend by currents flowing in the outer conductor.

*(Note: This screening of electric and magnetic fields is only perfect if the outer conductor has zero resistance. If we include a bit of resistance in the outer shield what happens? DC electric fields will still be screened: they will create static image charges in the shield metal. At high enough AC frequencies, the screening of the electric fields will fall off: the resistance will resist the movement of the image charges in the shield, and some AC electric field will leak out. What about magentic fields? At high frequencies, magnetic fields are screened well: Lenz's law creates large EMFs that can effectively drive currents in the outer shield to cancel the magnetic field outside the cable. At low frequencies, this no longer works: Consdider ramping ramping up a current in the center wire. At first, there will be an EMF that will create cancelling currents in the outer shield. When your ramp is done, the outer sheild then needs to carry a DC current. However, since there is no voltage driving the currents in the outer shield any more, if the outer shield has any resistance, these induced currents will dissipate and decay to zero. Upshot: the above picture holds also including resistance for a range of frequencies, but elecric shielding breaks down at high frequencies and magentic shielding breaks down at DC.)*

In the coaxial geometry, we can calculate exact analytical expressions from the capacitance per unit length $C' = C / \ell$, and the inductance per unit length $L' = L / \ell$:

$$
C' = \frac{2\pi \varepsilon}{\ln\!\left(\frac{D}{d}\right)}
\hspace{1in}
L' = \frac{\mu}{2\pi}\ln\!\left(\frac{D}{d}\right)
$$

The cable is typically filled with [PTFE](https://en.wikipedia.org/wiki/Polytetrafluoroethylene) (teflon), which has a dielectric constant $\epsilon = 2.1 \epsilon_0$ (reminder $\epsilon_0 = 8.85$ pF / m) and a magnetic permeability $\mu \approx \mu_0 = 1.25\ \mu$H / m. Using a plastic dielectric allows the cable to be flexibile while maintaining a fixed distance between the inner and outer conductor, ensuring that $C'$ and $L'$ are fixed (which, as we will see later, will be important for how signals propagate in the cable). For the typical [RG-58](https://en.wikipedia.org/wiki/RG-58) coaxial cable we use in lab, this will give $C' \approx 100$ pF / m and $L' = 250$ nH / m:

In [68]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread("figures/coax-kabel-rg58-impedantie-50-ohm.jpg")

fs=20

fig = plt.figure(figsize=(8,4))
plt.subplot(121)
plt.imshow(img)
plt.axis("off")
plt.subplot(122)
plt.axis("off")
plt.text(0.5,0.8,"RG-58:", fontsize=fs, ha='center')
plt.text(0.5,0.5,r"$C' = C / \ell \approx 100$ pF/m", fontsize=fs, ha='center')
plt.text(0.5,0.3,r"$L' = C / \ell \approx 250$ nH/m", fontsize=fs, ha='center')

show_plot(fig)


## Wave propagation in electrical circuits

### The Telegrapher's equations

Now that we know more about the properties of cables, let's see if we can contruct a circuit representation of what happens in a cable. To do this, we will build a [distributed-element model](https://en.wikipedia.org/wiki/Distributed-element_model) of our cable. We will break down the cable into small lengths $dx$ and represent these as lumped element circuits. From there, we will build equations based on Kirchoff's laws, and then taking the limit $dx \rightarrow 0$, we will obtain an exact result for how the cable will behave. 

Doing so, our equivalent circuit will look like this:

In [146]:
import schemdraw
import schemdraw.elements as elm

BLUE = '#2f6fed'

with schemdraw.Drawing(show=False) as d:
    d.config(unit=3, fontsize=10)

    inductors, caps, nodes, wires = [], [], [], []
    for vlbl in ['$v(x-dx,t)$', '$v(x,t)$', '$v(x+dx,t)$']:
        L = elm.Inductor().right().length(4).label("$L'dx$", loc='top', ofst=0.2)
        d += L
        inductors.append(L)
        d += (dot := elm.Dot())
        nodes.append(dot.end)
        x, y = dot.end
        d.push()
        d += elm.Label().at((x - 0.9, y + 1.1)).label(vlbl, color='b')
        d += elm.Line().at((x - 0.55, y + 0.95)).to((x - 0.05, y + 0.07)).color(BLUE).linewidth(1)
        d.pop()

        d.push()
        d += elm.Line().down().length(0.8)
        C = elm.Capacitor().down().length(0.8)
        d += C
        caps.append(C)
        d += elm.Label().at((dot.end[0] - 0.8, dot.end[1] - 1.07)).label("$C'dx$", fontsize=10)
        d += elm.Ground()
        d.pop()

    d += elm.Line().right().length(1)
    

    # Series currents (through 2nd and 3rd inductors) and capacitor currents
    d += elm.CurrentLabelInline(direction='in', color='r', ofst=1).at(inductors[1]).label('$i(x-dx,t)$',color='r')
    d += elm.CurrentLabelInline(direction='in', color='r', ofst=1).at(inductors[2]).label('$i(x,t)$', color='r')
    d += elm.CurrentLabelInline(direction='in', ofst=0.45).at(caps[0]).label(r'$i_c(x-dx,t)$')
    d += elm.CurrentLabelInline(direction='in', ofst=0.45).at(caps[1]).label(r'$i_c(x,t)$')
    d += elm.CurrentLabelInline(direction='in', ofst=0.45).at(caps[2]).label(r'$i_c(x+dx,t)$')
    d += elm.Label("...", fontsize=20).at((-0.5,-.1))
    d += elm.Label("...", fontsize=20).at((13.3,-.1))

    
#    d += elm.Label().at((nodes[-1][0] + 2.5, nodes[-1][1] - 1.2)).label('i → i(x,t)\nv → v(x,t)', fontsize=10)

display_schematic(d)

In this picture, the transmission line is divided into segements of length $dx$ labelled by position $x$ along the length of the cable.

For each segment, we define the node voltage at that position at given time $t$ as $v(x,t)$. At each node, there is an inductor of value $L'dx$ feeding current in from the left from the previous node at $x -dx$ , an inductor $L'dx$ feeding current into the next node at position $x + dx$ , and there is also a capacitor of value $C' dx$ that feeds current to ground. We define a function $i(x)$ that gives the current flowing in the center conductor as a function of position, and a function $i_c(x)$ that defines how much current is flowing to ground through the capacitors as each position along the line. 

We will now use Kirchoff's laws, and the current-voltage relationships of inductors and capacitors, to derive some equations.

Using $V = L dI/dt$, we will have:

$$
v(x+dx) - v(x) = - L' dx \frac{\partial i}{\partial t}
$$

Using Kirchoff's current law ($\sum I = 0$) at the node at position $x$, we find:

$$
i(x) - i(x-dx) + i_c(x) = 0
$$

Using $I_C = C (dV/dt)$ gives:

$$
i(x) - i(x-dx) = - C' dx \frac{\partial v}{\partial t}
$$

Taking the limit $dx \rightarrow 0$, we can rewrite these two equations as:

$$
\frac{\partial v}{\partial x} = - L' \frac{\partial i}{\partial t}
$$ (dvdx)

and

$$
\frac{\partial i}{\partial x} =  - C' \frac{\partial v}{\partial t}
$$ (didx)

Taking a partial derivative of {eq}(dvdx) with respect to time, interchanging the order of the partial derivatives, and then using {eq}(didx), we can find:

$$
\frac{\partial^2 v}{\partial x^2} 
= - L' \frac{\partial^2 i}{\partial x \partial t}
= - L' \frac{\partial^2 i}{\partial t \partial x}
=  L' \frac{\partial}{\partial t} \frac{\partial i}{\partial x}
= L' C' \frac{\partial^2 v}{\partial t^2}
$$

We can do the same with the second equation, leading to the following two equations for $v(x)$ and $i(x)$:

$$
\frac{\partial^2 v}{\partial t^2} =  \frac{1}{L' C'} \frac{\partial^2 v}{\partial x^2} 
\hspace{1in}
\frac{\partial^2 i}{\partial t^2} =  \frac{1}{L' C'} \frac{\partial^2 i}{\partial x^2} 
$$

You may recognise the form of these equations: they are both one-dimensional [wave equations](https://en.wikipedia.org/wiki/Wave_equation) with a wave phase velocity given by:

$$
v_p = \frac{1}{\sqrt{L'C'}}
$$

They are a lossess version of the [Telegrapher's equations](https://en.wikipedia.org/wiki/Telegrapher's_equations), a name that comes from their original use in predicting how signal propagate in [telegraph lines](https://en.wikipedia.org/wiki/Electrical_telegraph) in the 1800s. 

Note that these are electromagetic waves, corresponding solutions of Maxwell's equastion in the coaxial transmission line. For the RG-58 cables we looked at above, the phase velociy is given by $1.92 \times 10^8$ m/s, about 66% of the speed of light in vacuum. Why are waves slower in the cable than in free space? Of course, the boundary conditions are different, resulting in a different "effective stiffness" for the waves. The dielectric constant of the PTFE also plays a significant role ($1/\sqrt{\epsilon_r} = 1/\sqrt{2.1} \approx 0.69$), similar to how the dielectric contant of glass changes its refactive index, slowing down light even for regular plane waves. At this speed, waves in your cables will take about 5 nanoseconds to travel 1 meter: that is pretty fast, but close to the time resolution you can measure with the oscilloscope. With 10 meter long cable, this is actually relatively easy to measure, as you will see in the lab. 

### Propagation of wave fronts in a semi-infinite cable

Let's see how a coaxial cable responds to a "step response" of voltage, as we did for the RC, RL, and RLC circuits. As a reminder, we found that the current flowing into an RC circuit underwent a step followed by an exponential decay. The (R)LC circuits responded to a step response with a (damped) oscillating current. The coaxial cable transmission line model that we built above is a circuit that consists *only* of inductors and capacitors. What will it do? 

Specifically, we will consider a semi-infinite (really long) coaxial cable, extending from $x=0$ to $x = \infty$. At $t=0$, all currents and voltages in the cable are zero: $v(x,t<0) = 0$ and $i(x,t) = 0$. At $t=0$, we flip a switch that connects the end of the cable at $x=0$ to a voltage source. We then ask the question: what is $v(x,t)$? 

The answer is shown here below, which you should be familiar with from a course on waves:

In [206]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(13.7, 5.4))
ax.set_xlim(0, 13.7)
ax.set_ylim(0, 5.4)
ax.axis("off")

c = "k"          # everything except the data line
data_c = "b"     # data line of v in the bottom plot
lw = 1.8
arrow = dict(arrowstyle="->", color=c, lw=lw, shrinkA=0, shrinkB=0)

v_axis_len = 1.05   # height of the v axis (same in both plots)
v_level = 0.7       # height of V0 / the pulse above the baseline (same in both plots)

# ---------- top: source, switch, transmission line ----------
y0 = 4.85
ax.text(0.8, y0, r"$V_0$", fontsize=18, ha="center", va="center", color=c)
ax.plot([1.2, 1.65], [y0, y0], c=c, lw=lw)                  # wire from V0
ax.annotate("", xytext=(1.2, y0), xy=(1.5,y0), arrowprops=arrow)   # current in
plt.text(1.3,y0+0.2, "$I(t)$", fontsize=14)

ax.plot(1.65, y0, "o", c=c, ms=3)
ax.plot([1.65, 2.15], [y0, y0 + 0.25], c=c, lw=lw)          # open switch blade
ax.plot(2.2, y0, "o", c=c, ms=3)
ax.plot([2.2, 3.05], [y0, y0], c=c, lw=lw)                  # wire to line
ax.add_patch(plt.Circle((3.05, y0), 0.2, fill=False, ec=c, lw=lw))
ax.plot([3.05, 10.6], [y0 + 0.2, y0 + 0.2], c=c, lw=lw)       # transmission line (top)
ax.plot([3.05, 10.6], [y0 - 0.2, y0 - 0.2], c=c, lw=lw)       # transmission line (bottom)
ax.annotate("", xytext=(10.8, y0), xy=(11.2,y0), arrowprops=arrow)   # current in
ax.text(11.3, y0 + 0.0, r"$\infty$", fontsize=14, color=c, va="center")

# ---------- dashed vertical guide ----------
x_axis = 3.05
ax.plot([x_axis, x_axis], [y0, 3.65], c=c, ls=(0, (2, 4)), lw=1.2)
ax.plot([x_axis, x_axis], [2.6, 2.05], c=c, ls=(0, (2, 4)), lw=1.2)

# ---------- panel 1: t = 0 ----------
yb1 = 2.7                      # baseline
ytop1 = yb1 + v_level          # V0 level
ax.text(1.4, 3.2, r"$t = 0$", fontsize=14, color=c, va="center")
ax.annotate("", xy=(x_axis, yb1 + v_axis_len), xytext=(x_axis, yb1), arrowprops=arrow)   # v axis
ax.text(x_axis + 0.2, yb1 + v_axis_len, r"$v$", fontsize=16, color=c, va="center")
ax.annotate("", xy=(10.6, yb1), xytext=(x_axis, yb1), arrowprops=arrow)               # x axis
ax.text(10.8, yb1, r"$x$", fontsize=16, color=c, va="center")
ax.text(x_axis - 0.3, ytop1, r"$V_0$", fontsize=16, color=c, ha="right", va="center")
ax.plot([x_axis, x_axis], [yb1, ytop1], c=data_c, lw=2.5, zorder=5)                    # step
ax.plot(x_axis, ytop1, "o", c=data_c, ms=5, zorder=5)
ax.plot([x_axis, 9.4], [yb1, yb1], c=data_c, lw=2.5, zorder=5)

# ---------- panel 2: t = t1 ----------
yb2 = 0.95
ytop2 = yb2 + v_level
x1 = 8.0
ax.text(1.4, 1.4, r"$t = t_1$", fontsize=14, color=c, va="center")
ax.annotate("", xy=(x_axis, yb2 + v_axis_len), xytext=(x_axis, yb2), arrowprops=arrow)   # v axis
ax.text(x_axis + 0.2, yb2 + v_axis_len, r"$v$", fontsize=16, color=c, va="center")
ax.annotate("", xy=(10.6, yb2), xytext=(x_axis, yb2), arrowprops=arrow)               # x axis
ax.text(10.8, yb2, r"$x$", fontsize=16, color=c, va="center")
ax.text(x_axis - 0.3, ytop2, r"$V_0$", fontsize=16, color=c, ha="right", va="center")
ax.plot([x_axis, x1, x1, 9.5], [ytop2, ytop2, yb2, yb2], c=data_c, lw=2.5, zorder=5)  # pulse
ax.annotate("", xy=(8.8, 1.45), xytext=(8.2, 1.45), arrowprops=arrow)                 # direction
ax.text(9, 1.45, r"$v_p \sim 2\times10^{8}\ \mathrm{m/s}$", fontsize=12, color=c, va="center")
ax.text(x1, 0.25, r"$x_1 = t_1 / v_p$", fontsize=12, color=c, ha="center", va="center")
ax.plot([x1, x1], [0.5, 1], c=c, ls=(0, (2, 4)), lw=1.2)

show_plot(fig)

When we flip the switch at $t=0$, the voltage at the input of our coaxial cable jumps up to $V_0$. This then forms the boundary condition of our wave equation for $v(x,t)$. If you solve the wave equation for $v(x,t)$ for this initial condition, what you will find at is that what results is a "square" wave front of height $V_0$ that propagates along the cable at speed $v_p = 1/\sqrt{LC}$. After a time $t_1$, the square wave front has propagated a distance $x_1 = t_1 / v_p$ along the length of the cable.

What about the current $i(x,t)$? Since we know $v(x,t)$, we can use the earlier relations to calculate $i(x,t)$:

$$
\frac{\partial i}{\partial t} = - \frac{1}{L'} \frac{\partial v}{\partial x}
$$  

$$
\frac{\partial i}{\partial x} =  - C' \frac{\partial v}{\partial t}
$$ 

From the first equation, we can see that the current $i(x,t)$ will be constant every where for $x<x_1$. This also follows from the second equation. The result is that the current $i(x,t)$ is *also* a propagating square wave of height $I_0$.

How big is $I_0$? We can solve this mathematically by applying the boundary conditions to the above waves. But we can also solve this with some physical intuition: if you look at the propagating wave, since $v(x,t)$ is zero everywhere except at the wave front, we know that the current into the capacitors to ground everywhere is zero. What is physically happening is that the voltage source is suppling a current $I(t)$ which is being used to charge up the distributed capacitors of the transmission line as the voltage wave front propagates. The total charge on all the capacitors to ground at any given time is given by:

$$
Q(t) = \int_0^{x_1(t)} v(x) C' dx = V_0 C' x_1
$$

The current needed to supply this charge is then:

$$
I_0 = \frac{dQ}{dt} = V_0 C' \frac{dx_1}{dt} = V_0 C' v_p
$$

Using $v_p = \sqrt{1/(L'C')}$, we get:

$$
I_0 = V_0 \frac{C'}{\sqrt{C'L'}} = V_0 \sqrt{\frac{C'}{L'}} \equiv \frac{V_0}{Z_C}
$$

Here, we have defined the **characteristic impedance** of the cable $Z_C$ as:

$$
Z_C = \sqrt{\frac{L'}{C'}}
$$

The height of the propagating current wave front is determined by the ratio of the voltage being sourced to this characteristic impedance. 

OK, let's stop and think for a bit: if we have a semi-infinite transmission line, from the point of view of the voltage source, what does this mean? What the voltage source sees is that it has applied a constant voltage $V_0$ and what results is that it needs to supply a constant current $I_0$. If you did not know what you were connected to, you would think that the semi-inifinte coaxial cable looked like just a *regular resistor* with resitance $R_{\rm eff} = Z_C$! 

As crazy as it sounds, **this is also actually true!** Physically, what is happening is exactly what we described above: we launch a propagating square wave in the cable, it travels along the cable charging up all the capacitors at a constant rate, and doing so supplies a constant current. As long as the wave does not reach the other end, and as long as the wave does not reflect anywhere, then this is exactly what happens. In real life, cables are not infinitely long, so the above only holds in practice if you consider times $t$ shorter than it takes for the wave to reach the other end (5 ns per meter of cable), or if you do something special to make sure you don't get any reflection when it does reach the end (more on that later). 

What is the typical value of $Z_C$? This depends on the type of cable you are using. The RG-58 cable we use in the lab is engineered to have $Z_C = 50$ ohms: this is why the sticker and tags on the cables in your lab say 50 ohms! That value of 50 ohms is **not** the resistance of the wire inside the cable (this is on the order of 0.04 ohms per meter), the value of 50 ohms is independent of the length of the cable and is the value of the characteristic impedance $Z_C = \sqrt{L'/C'}$. It is the resistance *to ground* the cable would *appear* to have if it were infinitely long due to the charging of the capacitors by the propagating wave front.  

A final thought about the semi-inifinite cable: our ideal semi-infinite cable was constructed from only inductors and capacitors, neither of which can dissipate energy. But in the end, from the point of view of our voltage, our cable *looks* like a purely dissipative resistor! How can that be? The answer is that although it *looks* like the cable is dissipating energy, it is not: the energy is being *stored* in the capacitors as the wave propagates. However, if the cable is really semi-inifite, then there is no end to the number of capacitors we need to charge: the apparent power dissipation $P = (V_0)^2 / Z_C$ that the voltage source sees is not actually a rate of *dissipation* if energy but instead a rate of *storage* of energy in the electric fields in the capacitors of the cable. 

### What if the cable is not infinite? 

No cable in real life is actually infinite: what happens when the wave reaches the other end? 

In general, when the wave front reaches the end of the cable, like an open or clamped spring, the wave front will reflect and bounce back in the other direction to where it came from. 

What does this reflection look like? The answer depends on what is connected at the other end. There are three extreme cases that are useful to look at:

* **Open**: the end of the cable is an open-circuit (infinite impedeance) 
* **Matched**: the end of the cable is connected to ground (the shield) by a resistor with $R = Z_C$
* **Closed**: the end of the cable is short-circuited to ground (zero impedance)

Let's look first at the open case. In the open case, the current at the end of the cable is forced to zero, and the voltage is free to go up and down. The is analogous to a wave propagating in a string with an "open end" that can whip up and down. And if you look at the resulting $v(x,t)$, this is exactly what it looks like: 

In [209]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from matplotlib.patches import Ellipse

LW = 2.5
plt.rcParams["animation.embed_limit"] = 200   # MB; default 20 MB is too small for 360 frames
plt.rcParams.update({"axes.linewidth": LW, "xtick.major.width": LW, "ytick.major.width": LW, "lines.linewidth": LW, "font.size": 20, "mathtext.fontset": "dejavusans"})

# ---- parameters (normalised) ----
V0, L, vp = 1.0, 1.0, 1.0
tau = L / vp                      # one-way transit time
PERIOD = 4 * tau                  # v(x,t) is periodic with period 4 L/vp
N_PERIODS = 2
FPS, N_FRAMES = 30, 180*N_PERIODS    # two periods, 12 s loop
x = np.linspace(0, L, 1500)

def v(x, t, nmax=12):
    """Open end at x=L (Gamma=+1), ideal voltage source at x=0 (Gamma=-1)."""
    x = np.atleast_1d(x).astype(float)
    out = np.zeros_like(x)
    H = lambda s: (s >= 0).astype(float)
    for n in range(nmax):
        s = (-1) ** n
        out += s * H(vp*t - x - 2*n*L)
        out += s * H(vp*t - (2*(n+1)*L - x))
    return V0 * out

# ---- figure: 14 x 4 in, axes placed in inches ----
W, Hh = 14, 4
fig = plt.figure(figsize=(W, Hh))
def add_ax(l, b, w, h, **kw):
    return fig.add_axes([l/W, b/Hh, w/W, h/Hh], **kw)

LEFT, WID = 1.45, 7.3
ax_s = add_ax(LEFT, 2.45, WID, 1.45)       # schematic
ax   = add_ax(LEFT, 0.95, WID, 1.35, sharex=ax_s)   # v(x,t)
SQ = 2.75
ax_t = add_ax(10.55, 0.95, SQ, SQ)         # V(L,t), square
ax_t.set_box_aspect(1)

XL = (-0.32*L, 1.50*L)
ax.set_xlim(*XL)

# ---- schematic ----
ax_s.set_ylim(-1, 1); ax_s.axis("off")
h = 0.33
im = ax_s.imshow(np.zeros((1, len(x))), extent=(0, L, -h, h), aspect="auto",
                 cmap="viridis", vmin=0, vmax=2*V0, alpha=0.55, zorder=1)
for yy in (h, -h):
    ax_s.plot([0, L], [yy, yy], "k", lw=LW, zorder=3)
ax_s.plot([L, L], [-h, h], "k", lw=LW, zorder=3)
ax_s.plot([0, 0], [-h, h], "k", lw=LW, zorder=3)
ax_s.plot([0, 1.18*L], [0, 0], color="firebrick", lw=LW, zorder=4)
ax_s.plot(1.18*L, 0, "o", mfc="white", mec="firebrick", ms=9, mew=LW, zorder=5)
ax_s.text(1.23*L, 0, "open", va="center", ha="left", fontsize=20)

fig.canvas.draw()
bb = ax_s.get_window_extent()
ux = (XL[1]-XL[0]) / bb.width
uy = 2.0 / bb.height
d = 0.62 * fig.dpi * 0.85
cx = -0.19*L
ax_s.add_patch(Ellipse((cx, 0), d*ux, d*uy, fc="white", ec="k", lw=LW, zorder=3))
ax_s.plot([cx+d*ux/2, 0], [0, 0], color="firebrick", lw=LW, zorder=4)
ax_s.plot([cx, cx], [-d*uy/2, -0.88], "k", lw=LW)
ax_s.plot([cx, 0], [-0.88, -0.88], "k", lw=LW)
ax_s.plot([0, 0], [-0.88, -h], "k", lw=LW)
ax_s.text(cx, 0.0, "$V_0$", ha="center", va="center", fontsize=20, zorder=5)
ax_s.text(L/2, h+0.08, "coax", ha="center", va="bottom", fontsize=20)
ax_s.set_xlim(*XL)

# ---- v(x,t) ----
ax.axvspan(XL[0], 0, color="0.95", zorder=0)
ax.axvspan(L, XL[1], color="0.95", zorder=0)
ax.axvline(0, color="0.5", lw=LW); ax.axvline(L, color="0.5", lw=LW)
line, = ax.plot([], [], color="C0", lw=LW)
ax.set_ylim(-0.3*V0, 2.3*V0)
ax.axhline(0, color='0.4', ls='--', lw=LW, zorder=1)
ax.set_xticks([0,L]); ax.set_xticklabels(["0", "$L$"])
ax.set_yticks([0, V0, 2*V0]); ax.set_yticklabels(["0", "$V_0$", "$2V_0$"])
ax.set_xlabel("position $x$", labelpad=2); ax.set_ylabel("$v(x,t)$", labelpad=2)
ttxt = ax.text(1.03*L, 1.0*V0, "", ha="left", va="center", fontsize=20)
for s in ("top", "right"): ax.spines[s].set_visible(False)

# ---- V(L,t), 1:1 ----
TT = N_PERIODS*PERIOD
tt = np.linspace(0, TT, 4000, endpoint=False)
vL = np.array([v(L, t)[0] for t in tt])
ax_t.plot(tt/tau, vL, color="C0", alpha=0.25, lw=LW)
prog, = ax_t.plot([], [], color="C0", lw=LW)
dot,  = ax_t.plot([], [], "o", color="C0", ms=10)
ax_t.set_xlim(0, TT/tau); ax_t.set_ylim(-0.3*V0, 2.3*V0)
ax_t.set_xticks([0, 4, 8])
ax_t.axhline(0, color='0.4', ls='--', lw=LW, zorder=0); ax_t.set_yticks([0, V0, 2*V0]); ax_t.set_yticklabels(["0", "$V_0$", "$2V_0$"])
ax_t.set_xlabel(r"$t/t_0$", labelpad=2); ax_t.set_ylabel("$v(L,t)$", labelpad=2)
for s in ("top", "right"): ax_t.spines[s].set_visible(False)

def update(i):
    t = i * TT / N_FRAMES
    vx = v(x, t)
    line.set_data(x, vx); im.set_data(vx[None, :])
    ttxt.set_text(f"$t={t/tau:.2f}\\,t_0$")
    m = tt <= t
    prog.set_data(tt[m]/tau, vL[m])
    dot.set_data([t/tau], [v(L, t)[0]])
    return line, im, ttxt, prog, dot

ani = FuncAnimation(fig, update, frames=N_FRAMES, blit=False)
plt.close(fig)                       # avoid a duplicate static figure in the output
HTML(ani.to_jshtml(fps=FPS, default_mode="loop"))

The voltage source is configured to generate a step response at $t=0$:

$$
V_s(t) = 
\left\{ 
\begin{array}{ll} 
& 0 \hspace{0.5cm} & t <  0 \\
& V_0 \hspace{0.5cm} & t \geq 0 
\end{array}
\right.
$$

In the animation, first, you can see the initial launching of the wave front by the voltage source. Like in the semi-infinite cable, the propagates at a speed $v_p$ close to the speed of light, reaching the other side of the cable of length $L$ at a time $t_0 = L / v_p$. 

Once it reaches the open end of the cable, it reflects. For an open circuit boundary condition, the wave will reflect with a reflection coefficient $\Gamma = 1$: the reverse propagating wave will also be a wave with a step $\Delta V = +V_0$. Interestingly, the voltage you would measure if you put a voltage meter at the other end of the cable is actually *twice* as large as is supplied by the source to start with due to the contructive interference of the two waves! 

This reflected wave then propgages backwards towards the voltage source. If you remember the previous lectures, you will know that the (ideal) voltage source presents an input impedance of zero ohms: it looks like a short circuit. At a short circuited end, the reflection coefficient of the wave is $\Gamma = -1$: the wave front that bounces off the voltage source will have a step of $\Delta V = -V_0$. As it propagates it will bring the voltage inside the cable back down to $V_0$. 

When the wave front reaches the open end again, it will again reflect from the open end with $\Gamma = 1$: however, since the step of this new incoming wave is $-V_0$, it will turn into a step downwards propagating downwards, bringing the voltage in the cable back to zero as it propages. It will again reach the voltage source, the wave will invert, now constructively interfering to bring the voltage back up to $2 V_0$ again. Note that the voltage on the open end will undergo a square wave osillation in time from 0 to $2 V_0$, with a period given by $4 t_0$ (frequency $f = 1/(4 t_0$)). A bit like the series LC circuit, the short-open coaxial cable will also "ring" like a bell if you hit is with a hammer (square voltage pulse), with the difference now that the oscillations are a square wave rather than a sine wave (like an echo in a tunnel). If there is no loss (zero resistance in the wires of the cables, an ideal open circuit at one end, and an ideal closed circuit at the other), this bouncing back and forth will go on forever!

What if the impedance of the two ends is not zero and not infinity? For a resistance $R_L$ at the end of the cable, you will then have a reflection coefficient given by:

$$
\Gamma = \frac{R_L - Z_C}{R_L + Z_C}
$$

From this you can already see back the two limits mentioned above: for $R_L = 0$, we get:

$$
\Gamma_{\rm short} = \frac{- Z_C}{Z_C} = -1
$$

For $R \rightarrow \infty$, $(R_L - Z_C) \rightarrow R$ and $(R_L + Z_C) \rightarrow R$ and then then we get the limit:

$$
\Gamma_{\rm open} = \frac{R_L}{R_L} = 1
$$

### Impedance Matching

The third special case is if we choose $R_L = Z_C$, in which case we get:

$$
\Gamma = \frac{Z_C - Z_C}{Z_C + Z_C} = 0
$$

If the impedance of the thing we attach at the end of our cable also looks like a resistor with a resistance equal to $Z_C$, then **there are no reflection at all!** This condition of [**impedance matching**](https://en.wikipedia.org/wiki/Impedance_matching) makes life **much** easier when you need to deal with high frequency signals, whose wavelength becomes smaller than the length of the cable! If you are dealing with "high" frequency signals, then you always want the devices you connect to your cables to be present the same impedance as the characteristic impedance of the cable you are using. If not, then when you launch a signal down the cable, like our voltage step above, you will not only see the initial voltage step sent out, but also "echos" of it from the reflections bouncing back and forth in the cable. To avoid these echos, the output impedance of your sources, and the input impedance of your voltage meters, and of any connection you make to your cable, must *all* present an impedance of $Z_C$ to the cable. The biggest challenge in RF and microwave engineering is ensuring that every part of the path the signal travels down presents the same impedance in order to prevent reflections and standing waves. If we match teh end of the cable with an impedance equal to $Z_C$ to prevent reflections, then we say that the cable has been properly [terminated](https://en.wikipedia.org/wiki/Electrical_termination). 


Note that for high enough frquencies, it is not as simple as just soldering a 50 ohm resistor on the end of your cable: if the wires that connect the cable to the resistor are comparble to the wavelenth of the frequency you are intersted in ($\lambda \approx 2$  cm for 1 GHz), then reflections can become important and you can get standing wave resonances (more on that below): a good rule of thumb is that for any wire longer than $\lambda/10$ (2 mm at 1 GHz) then you should make sure that your connecting wires should also have the right $C'$ and $L'$ to have a 50 ohm impedance. For coaxial cables, special [terminator](https://en.wikipedia.org/wiki/Electrical_termination#/media/File:10base2_BNC_Terminator.jpg) devices are engineered to provide a high frequency 50 ohm impedance wiring connection to a 50 ohm resistor to make sure that no waves reflect.

This now *also* explains why the waveform generator you use in the lab that we looked at in Lecture 2 is not designed to be an ideal voltage source (which would have a zero ohm output impedance) but instead engineered tto have a 50 ohm output impedance! This so that the output of the waveform generator is matched to the (lab standard) 50 ohm coaxial cable that we have all chosen to use, and therefore preventing any reflections from waves bouncing back into the generator. 

### RC time and the ring-up staircase of a cable

As an illustration of how a resistance different that $Z_C$ affects reflections, let's consider adding a series resistance to the voltage source. If we ignore any wave propagation in the cable and consider only its total capacitance, then this series resistance would result in a low-pass filtering of the sharp voltage step from the RC time. How does this change if we consider the propagation of the wave front? Let's have a look:

In [214]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import Ellipse

LW = 2.5
plt.rcParams["animation.embed_limit"] = 300   # MB; default 20 MB is far too small
plt.rcParams.update({"axes.linewidth": LW, "xtick.major.width": LW, "ytick.major.width": LW,
                     "lines.linewidth": LW, "font.size": 20, "mathtext.fontset": "dejavusans"})

# ---- physics (normalised): open end at x=L, source with series resistance Rs = 10 Z0 ----
V0, L, vp = 1.0, 1.0, 1.0
t0 = L / vp                               # one-way transit time
Rs_over_Z0 = 10.0
G_L = 1.0                                  # open circuit
G_S = (Rs_over_Z0 - 1) / (Rs_over_Z0 + 1)  # source reflection coefficient = 9/11
G = G_L * G_S                              # round-trip factor
V1 = V0 / (1 + Rs_over_Z0)                 # launched step = V0 Z0/(Rs+Z0) = V0/11
N_BOUNCE = 10                              # bounces at each end
T_END = 2 * N_BOUNCE * t0 + 1.0 * t0       # 10 bounces at each end, + 1 t0 to see the tail
FPS, PER_T0 = 20, 12      # fewer frames keeps the embedded notebook output small
N_FRAMES = int(T_END / t0 * PER_T0)
x = np.linspace(0, L, 1500)

def v(x, t, kmax=N_BOUNCE + 2):
    """Sum of forward and backward step fronts (bounce-diagram solution)."""
    x = np.atleast_1d(x).astype(float)
    out = np.zeros_like(x)
    H = lambda s: (s >= 0).astype(float)
    for k in range(kmax):
        out += G**k * H(vp*t - x - 2*k*L)                      # forward front k
        out += G_L * G**k * H(vp*t - (2*(k+1)*L - x))          # backward front k
    return V1 * out

# ---- figure: stacked layout, axes placed in inches ----
W, Hh = 10, 9.3
fig = plt.figure(figsize=(W, Hh))
def add_ax(l, b, w, h, **kw):
    return fig.add_axes([l/W, b/Hh, w/W, h/Hh], **kw)
LEFT, WID, PH = 1.45, 8.2, 1.35
ax_t = add_ax(LEFT, 0.95, WID, 3*PH)               # v(L,t) -- three times as tall as v(x,t)
ax   = add_ax(LEFT, 6.05, WID, PH)                 # v(x,t)
ax_s = add_ax(LEFT, 7.50, WID, 1.70, sharex=ax)    # schematic
XL = (-0.45*L, 1.45*L)   # equal grey margins left and right of the line
ax.set_xlim(*XL)
YL = (-0.1*V0, 1.1*V0); VMAX = V0

# ---- schematic ----
ax_s.set_ylim(-1, 1); ax_s.axis("off")
h, yb = 0.33, -0.88
im = ax_s.imshow(np.zeros((1, len(x))), extent=(0, L, -h, h), aspect="auto",
                 cmap="viridis", vmin=0, vmax=VMAX, alpha=0.55, zorder=1)
for yy in (h, -h):
    ax_s.plot([0, L], [yy, yy], "k", lw=LW, zorder=3)
ax_s.plot([L, L], [-h, h], "k", lw=LW, zorder=3)
ax_s.plot([0, 0], [-h, h], "k", lw=LW, zorder=3)
ax_s.plot([0, 1.18*L], [0, 0], color="firebrick", lw=LW, zorder=4)
ax_s.plot(1.18*L, 0, "o", mfc="white", mec="firebrick", ms=9, mew=LW, zorder=5)
ax_s.text(1.23*L, 0, "open", va="center", ha="left", fontsize=20)

fig.canvas.draw()
bb = ax_s.get_window_extent()
ux = (XL[1]-XL[0]) / bb.width; uy = 2.0 / bb.height
d = 0.62 * fig.dpi * 0.85
cx = -0.36*L
ax_s.add_patch(Ellipse((cx, 0), d*ux, d*uy, fc="white", ec="k", lw=LW, zorder=3))
# series resistor between source and centre conductor
xa, xb = cx + d*ux/2, 0.0
x1, x2 = xa + 0.03*L, xb - 0.03*L
ax_s.plot([xa, x1], [0, 0], color="firebrick", lw=LW, zorder=4)
ax_s.plot([x2, xb], [0, 0], color="firebrick", lw=LW, zorder=4)
zx = np.linspace(x1, x2, 9); zy = 0.2*np.array([0,1,-1,1,-1,1,-1,1,0])
ax_s.plot(zx, zy, color="firebrick", lw=LW, zorder=4)
ax_s.text((x1+x2)/2, 0.38, "$10Z_C$", ha="center", va="bottom", fontsize=20)
ax_s.plot([cx, cx], [-d*uy/2, yb], "k", lw=LW)
ax_s.plot([cx, 0], [yb, yb], "k", lw=LW)
ax_s.plot([0, 0], [yb, -h], "k", lw=LW)
ax_s.text(cx, 0.0, "$V_0$", ha="center", va="center", fontsize=20, zorder=5)
ax_s.text(L/2, h+0.08, "coax", ha="center", va="bottom", fontsize=20)
ax_s.set_xlim(*XL)

# ---- v(x,t) ----
ax.axvspan(XL[0], 0, color="0.95", zorder=0); ax.axvspan(L, XL[1], color="0.95", zorder=0)
ax.axvline(0, color="0.5", lw=LW); ax.axvline(L, color="0.5", lw=LW)
ax.axhline(0, color="0.4", ls="--", lw=LW, zorder=1)
line, = ax.plot([], [], color="C0", lw=LW)
ax.set_ylim(*YL)
ax.set_xticks([0, L/2, L]); ax.set_xticklabels(["0", "$L/2$", "$L$"])
ax.set_yticks([0, V0]); ax.set_yticklabels(["0", "$V_0$"])
ax.set_xlabel("position $x$", labelpad=2); ax.set_ylabel("$v(x,t)$", labelpad=2)
ttxt = ax.text(1.03*L, 0.5*V0, "", ha="left", va="center", fontsize=20)
for s in ("top", "right"): ax.spines[s].set_visible(False)

# ---- v(L,t), same size as v(x,t) ----
tt = np.linspace(0, T_END, 6000)
vL = np.array([v(L, t)[0] for t in tt])
ax_t.axhline(0, color="0.4", ls="--", lw=LW, zorder=0)
ax_t.plot(tt/t0, vL, color="C0", alpha=0.25, lw=LW)
# lumped-capacitor charging through Rs: C = C'L = t0/Z0, so RC = Rs*C = 10 t0
RC = Rs_over_Z0 * t0
ax_t.plot(tt/t0, V0*(1 - np.exp(-tt/RC)), color="0.25", lw=LW, zorder=2)
ax_t.text(21.0, 0.30*V0, r"$V_0\,(1-e^{-t/RC}),\ RC=10\,t_0$", color="0.25", ha="right", va="center", fontsize=20)
prog, = ax_t.plot([], [], color="C0", lw=LW)
dot,  = ax_t.plot([], [], "o", color="C0", ms=10)
ax_t.set_xlim(0, T_END/t0); ax_t.set_ylim(*YL)
ax_t.set_xticks([0, 5, 10, 15, 20]); ax_t.set_yticks([0, V0]); ax_t.set_yticklabels(["0", "$V_0$"])
ax_t.set_xlabel(r"$t/t_0$", labelpad=2); ax_t.set_ylabel("$v(L,t)$", labelpad=2)
for s in ("top", "right"): ax_t.spines[s].set_visible(False)

def update(i):
    t = i * T_END / (N_FRAMES - 1)
    vx = v(x, t)
    line.set_data(x, vx); im.set_data(vx[None, :])
    ttxt.set_text(f"$t={t/t0:.2f}\\,t_0$")
    m = tt <= t
    prog.set_data(tt[m]/t0, vL[m]); dot.set_data([t/t0], [v(L, t)[0]])
    return line, im, ttxt, prog, dot

ani = FuncAnimation(fig, update, frames=N_FRAMES, blit=False)
plt.close(fig)                       # avoid a duplicate static figure below the animation

scale = 0.7
HTML(f'''
<div style="display:flex; justify-content:center; width:100%">
  <div style="zoom:{scale}">
    {ani.to_jshtml(fps=FPS, default_mode="loop")}
  </div>
</div>
''')

Like before, the step from the voltage source launches a propagating wave, but now one with a smaller amplitude give by $V_0 \frac{Z_C}{R}$. This reflects off of the open end, travelling back to the source. At the source, it is then *partially* reflected with $\Gamma = \frac{9}{11} = 0.911$, coming back with less amplitude each time. 

In the plot, you can see the exponential plot you would get if you ignored the inductance of the cable and looked at only the RC time. Aside from the staircase of steps from the propagating waves, it looks very similar: the steps from the reflections reconstruct the exponential growth of the signal! 

As a final comment on this before we move on: in the RC approximation, we have filtering of the signal, reducing its bandwidth to the RC time: the capacitance of the cable dramatically reduces our bandwidth. However, if you look at only the first step edge propagating in the cable, you can see that the full bandwidth of the sharp step is fully intact: it is delayed in time by the propagation time in the cable, and is reduced in amplitude, but contains all the Fourier components and full bandwidth of the original signal. If we terminated the cable with $Z_C$ instead of leaving if open, the cable capacitance would not affect the bandwidth of our signal at all: it would be identical in shape to the step we started with. 

### Complex reflection coefficients

What if you terminate you cable with a inductor or a capacitor? 

An inductor or capacitor would present a different boundary condition to our wave equation. The easiest way to analyze this is to define a frequency-dependent complex-valued reflection coefficient, replacing the load resistance by a load impedance:

$$
\Gamma (\omega) = \frac{Z_L(\omega) - Z_C}{Z_L(\omega) + Z_C}
$$

What does this mean physically? To use it, we first write our incoming wave in phasor notation with a complex amplitude $\tilde V_{\rm in}(\omega)$ for frequency $\omega$:

$$
V_{\rm in} = \tilde V_{\rm in}(\omega) e^{j(\omega t - k x)}
$$

where the wave number $k$ is given by the ratio of the angular frequency adn the phase velocity: $k = \frac{\omega}{v_p}$. After the reflection, there will be a outgoing wave travelling in the opposite direction:

$$
V_{\rm out} = \tilde V_{\rm out}(\omega) e^{j(\omega t + k x)}
$$

The complex reflection coefficient tells us the ratio of the phasors of the two waves:

$$
\tilde V_{\rm out}(\omega) = \Gamma(\omega) \tilde V_{\rm in}(\omega)
$$

Note that if you have an inductor or a capacitor at the end, the reflected wave for a given frequency will not only have a different amplitude, but can have a (frequency dependent) phase shift. To apply this to a wave defined by a time dependence $v(x,t)$ like we had above, we first have to translate $v(x,t)$ into the Fourier domain by taking a Fourier transform (more on this later in the course). 

Finally, we have considered here what happens for reflection off of the end of a cable. What if you have another cable aftewards? Or multiple cables coming together? The reflection coefficient we calculate above is actually a specific example of a more general framework of [network analysis](https://en.wikipedia.org/wiki/Network_analysis_(electrical_circuits)), a formalism that allows one to calculate the scattering parameters ("S-parameters") of arbitrary electrical networks, the reflection coefficient above being the case for $S_{11}$ (the scattering parameter back into port 1 back from waves entering on port 1).

## Standing wave resonances in non-terminated cables

If you have cables that are not terminated at either end, then you can build up electrical standing wave resonances in the cable, like the vibrations of a guitar string or the acoustic sound waves in a flute. 

Exactly what frequencies these resonances occur at depends on the length of the cable and the boundary conditions on each end. If we have a cable that is open at both ends, you end up with $\lambda/2$ resonances:

In [249]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

L, MODE = 1.0, 1                  # length, mode number n
C_V, C_I = "#D55E00", "#0072B2"   # voltage, current colours
TITLE_FS = 12   # font size for titles and x-axis labels
x = np.linspace(0, L, 500)
kh, kq = MODE*np.pi/L, (2*MODE-1)*np.pi/(2*L)

cases = [
    dict(title=r'"$\lambda/2$" resonator', ends=("open", "open"), v=np.cos(kh*x), i=np.sin(kh*x),
         eq=r"$\lambda_n=\dfrac{2L}{n}$,   $f_n=\dfrac{n}{2}\dfrac{v_p}{L}$"),
    dict(title=r'"$\lambda/4$" resonator', ends=("open", "short"), v=np.cos(kq*x), i=np.sin(kq*x),
         eq=r"$\lambda_n=\dfrac{4L}{2n-1}$,   $f_n=\dfrac{2n-1}{4}\dfrac{v_p}{L}$"),
    dict(title=r'"$\lambda/2$" resonator', ends=("short", "short"), v=np.sin(kh*x), i=np.cos(kh*x),
         eq=r"$\lambda_n=\dfrac{2L}{n}$,   $f_n=\dfrac{n}{2}\dfrac{v_p}{L}$"),
]

def draw_cable(ax, ends):
    # same x-limits as the plot below, so cable ends line up with x=0 and x=L
    ax.set_xlim(0, 1.04*L); ax.set_ylim(-1, 1); ax.axis("off")
    kw = dict(clip_on=False)
    ax.add_patch(plt.Rectangle((0, -0.35), L, 0.7, fill=False, lw=1.5, ec="k", **kw))
    ax.plot([0, L], [0, 0], color="0.6", lw=1, ls=":", **kw)
    for xe, side, kind in [(0, -1, ends[0]), (L, 1, ends[1])]:
        if kind == "open":
            ax.plot([xe, xe+side*0.08], [0, 0], "k", lw=1.5, **kw)
            ax.plot(xe+side*0.08, 0, "o", color="k", ms=6, **kw)
        else:
            ax.plot([xe, xe+side*0.06, xe+side*0.06, xe], [0, 0, -0.35, -0.35], "k", lw=1.5, **kw)
            ax.plot(xe+side*0.06, 0, "o", color="k", ms=6, **kw)   # solid dot where wire leaves the coax
        ax.text(xe, 0.6, kind, ha="center", fontsize=12)

def plot_case(ax_c, ax, c):
    """Draw one case: cable schematic on ax_c, standing waves on ax."""
    draw_cable(ax_c, c["ends"])
    ax.plot(x, c["v"], color=C_V, lw=2, label="voltage $v(x)$")
    ax.plot(x, c["i"], color=C_I, lw=2, label="current $i(x)$")
    # curve labels, coloured like the curves, placed on the side facing away from the other curve
    xl = int(0.85 * len(x))
    for name, y, y_other, col in [("v", c["v"], c["i"], C_V), ("i", c["i"], c["v"], C_I)]:
        up = y[xl] > y_other[xl] or abs(y[xl]) < 0.35   # avoid sitting on the zero line
        ax.text(x[xl], y[xl] + (0.1 if up else -0.1), f"${name}(x)$", color=col,
                fontsize=TITLE_FS, ha="center", va="bottom" if up else "top")
    ax.axhline(0, color="k", lw=1)
    for xv in (0, L):
        ax.axvline(xv, color="0.4", ls="--", lw=1)
    ax.set(xlim=(0, 1.04*L), ylim=(-1.35, 1.35), yticks=[], xticks=[0, L])
    ax.set_xticklabels([r"$x=0$", r"$x=L$"], fontsize=TITLE_FS)
    ax.tick_params(axis="x", length=0)
    # title below the plot, centred between x=0 and x=L (axes x-range is 0..1.04 L)
    ax.set_xlabel(c["title"] + "   " + c["eq"], fontsize=16, labelpad=LABEL_PAD, x=0.5/1.04)
    for sp in ax.spines.values(): sp.set_visible(False)
    if abs(c["v"][0]) > 0.5:
        a1, a2 = "voltage antinode", "current node"
    else:
        a1, a2 = "current antinode", "voltage node"
    ax.annotate(a1, xy=(0, 1), xytext=(0.12, 1.2), arrowprops=dict(arrowstyle="->"), fontsize=10)
    ax.annotate(a2, xy=(0, 0), xytext=(0.12, -1.1), arrowprops=dict(arrowstyle="->"), fontsize=10)

CASE = 0   # 0: open-open (λ/2), 1: open-short (λ/4), 2: short-short (λ/2)
CABLE_SHIFT = -0.05   # figure-fraction to move cable drawing up
LABEL_PAD = 20       # distance between x tick labels and the title text

fig = plt.figure(figsize=(7.5, 4.2))
gs = fig.add_gridspec(2, 1, height_ratios=[0.6, 2], hspace=0.3)
ax_c, ax = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
plot_case(ax_c, ax, cases[CASE])
p = ax_c.get_position()
ax_c.set_position([p.x0, p.y0 + CABLE_SHIFT, p.width, p.height])  # move cable up

show_plot(fig)

For the case that both ends are open, we end up with current nodes, and voltage antinodes, at both ends of the cable. Resonances occur at wavelengths $\lambda_n$ and frequencies $f_n$ given by:

$$
\lambda_n = \frac{2L}{n}
\hspace{1in}
f_n = \frac{n}{2} \frac{v_p}{L}
$$

The first one is at the condition $L = \lambda/2$, then $\lambda$, then $3\lambda/2$, and so forth, spaced by frequency $\Delta f = v_p / 2L$. The lowest frequency mode is known as the [fundamental mode](https://en.wikipedia.org/wiki/Fundamental_frequency), and the higher frequencies are know as [harmonics](https://en.wikipedia.org/wiki/Harmonic). 

We can also consider the case of two shorted ends:

In [251]:
fig = plt.figure(figsize=(7.5, 4.2))
gs = fig.add_gridspec(2, 1, height_ratios=[0.6, 2], hspace=0.3)
ax_c, ax = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
plot_case(ax_c, ax, cases[2])
p = ax_c.get_position()
ax_c.set_position([p.x0, p.y0 + CABLE_SHIFT, p.width, p.height])  # move cable up

show_plot(fig)

This also forms $\lambda/2$ resonances, but now with current antinodes, and voltage nodes, at the ends of the cable. 

The last case we can consider is one open and one shorted end:

In [252]:
fig = plt.figure(figsize=(7.5, 4.2))
gs = fig.add_gridspec(2, 1, height_ratios=[0.6, 2], hspace=0.3)
ax_c, ax = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
plot_case(ax_c, ax, cases[1])
p = ax_c.get_position()
ax_c.set_position([p.x0, p.y0 + CABLE_SHIFT, p.width, p.height])  # move cable up

show_plot(fig)

Here, we end up with a voltage antinode at the open end and a voltage node at the shorted end, and vice versa for the. This is known as a $\lambda/4$ resonator, it has resonances at $\lambda/4$, $3\lambda/4$, $5\lambda/4$, and so forth. The harmonics are also spaced by $\Delta f = v_p / 2L$, but now start at $f_1 = v_p/4L$. 

### Adding a series resistance: Damping

Although the resistances are small, they are not zero. Say the wire in our cable has a resistance per unit length of $R'$. What happens if we add them into our model? 

In [257]:
import schemdraw
import schemdraw.elements as elm

with schemdraw.Drawing(show=False) as d:
    d.config(unit=3, fontsize=14)

    d += elm.DotDotDot()                      # line continues to the left

    for i in range(2):                        # change range() for more sections
        d += elm.Resistor().right().label("$R'dx$", loc='top')
        d += elm.Inductor().right().label("$L'dx$", loc='top')
        d += elm.Dot()
        d.push()
        d += elm.Capacitor().down().label("$C'dx$", loc='bottom')
        d += elm.Ground()
        d.pop()

    d += elm.Line().right().length(1)
    d += elm.DotDotDot()                      # line continues to the right

display_schematic(d)

I will not go through the math, but as long as the cable is "low loss" and satisfies the condition that $R' \ll omega L'$, then what results is that waves still propage but are also attenuated. For an outgoing wave with amplitude $v_0$ at $x=0$, we will then have:

$$
v = v_0 e^{-\alpha x} e^{j(\omega t - kx )}
$$

The term $\alpha$ is the **attenuation**, typically quoted in dB/meter for a cable. In the low-loss limit, the attenuation is given by:

$$
\alpha \approx \frac{R'}{2 Z_0}
$$

As a concrete example, the RG58 cable from the lab has an attenuation of 0.15 dB/m at 100 MHz. 

While the formula above is nice, in practice in real life, the model is not so simple. First of all, there will be some (frequency dependent) dilectric losses in the PTFE, which are also not negligible. Second, due to the [skin effect](https://en.wikipedia.org/wiki/Skin_effect), the effective resistance of the center conductor increase at high frequencies due to crowiding of the current at the edges of the wire from eddy currents. The crowding reduces the area of the center conductor, increasing its resistance. This effect scales with $\sqrt{\omega}$. In fact, for high frequencies, almost all the current flows in the outer 10 microns of the metal wire. Often, a cheap and poor conductive wire is used for the center conductor which is then coated in copper or silver. 



### Applications of transmission line theory 

Let's ask Gemini where the theory we discussed is relevant for applied physics:

* **Superconducting Qubits & Circuit QED:** Modeling coplanar waveguides (CPWs), quarter- and half-wavelength readout resonators, Purcell filters, and cryogenic microwave control/readout lines coupled to superconducting qubits (such as transmons).
* **RF & Microwave Engineering:** Designing impedance-matching networks (quarter-wave transformers, single- and double-stub tuners), Wilkinson power dividers, directional couplers, and distributed microwave filters using microstrip, stripline, and coaxial lines.
* **High-Speed Digital Interconnects:** Analyzing signal integrity, reflections, ringing, crosstalk, and propagation delay on multi-gigabit PCB traces, packages, and chip-to-chip links when electrical length is non-negligible compared to signal rise/fall time.
* **Time-Domain Reflectometry (TDR) & Dielectric Sensing:** Detecting open/short faults, kinks, or impedance discontinuities in cables and waveguides, and inferring local dielectric permittivity or moisture content from reflection coefficients and round-trip delay.
* **Pulsed Power & Accelerator Systems:** Using charged transmission lines, pulse-forming networks (PFNs), and Blumlein lines to shape high-voltage, nanosecond-scale rectangular pulses for kicker magnets, Marx-style pulse systems, and particle accelerators.
* **Antenna Feed Networks & Traveling-Wave Structures:** Designing balanced-to-unbalanced transitions (baluns), impedance-matched feedlines, and traveling-wave or leaky-wave antennas where phase velocity along a guiding line governs radiation angle and bandwidth.
* **Fusion & Plasma RF Coupling:** Modeling coaxial, waveguide, and transmission-line feeding structures that deliver ion cyclotron (ICRH) or lower-hybrid RF power into magnetized plasmas and match varying plasma boundary impedances.
* **Electrochemical & Bioelectrical Cable Models:** Applying lossy RC transmission-line equations (the telegrapher's equations in the low-frequency/inductance-neglected limit) to passive signal propagation along neuronal dendrites/axons and distributed impedance in porous electrodes..